# 00 - Zentrales Preprocessing (Gutenberg Gait Database)

In [1]:
import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

## Konfiguration

In [ ]:
DATA_PATH = 'GutenbergGaitDatabase/'
RAW_FORCE_FILE = 'GRF_F_V_PRO_left.csv'   # vertikale Kraftrichtung
AP_FORCE_FILE = 'GRF_F_AP_PRO_left.csv'   # anterior-posterior
ML_FORCE_FILE = 'GRF_F_ML_PRO_left.csv'   # medio-lateral
METADATA_FILE = 'GRF_metadata.csv'

OUTPUT_DIR = 'saved_models/prepared'
PREPARED_DATA_FILE = 'prepared_data.csv'
SPLIT_META_FILE = 'split_meta.json'

TEST_SIZE = 0.2
RANDOM_STATE = 42

# So viele Age-/Height-Brackets wie möglich (bis MAX), aber jede Klasse
# braucht mindestens MIN_SUBJECTS_PER_BRACKET Subjekte.
AGE_MIN_SUBJECTS_PER_BRACKET = 30
AGE_MAX_BRACKETS = 4
HEIGHT_MIN_SUBJECTS_PER_BRACKET = 30
HEIGHT_MAX_BRACKETS = 4

COL_SUBJECT_ID = 'SUBJECT_ID'
COL_SESSION_ID = 'SESSION_ID'
COL_TRIAL_ID = 'TRIAL_ID'
COL_DATASET_ID = 'DATASET_ID'
COL_AGE = 'AGE'
COL_SEX = 'SEX'
COL_HEIGHT = 'HEIGHT'
COL_BODY_MASS = 'BODY_MASS'
COL_SESSION_TYPE = 'SESSION_TYPE'
BASELINE_SESSION_TYPE_VALUE = 1

EXPECTED_META_COLS = [COL_AGE, COL_SEX, COL_HEIGHT, COL_BODY_MASS,
                       COL_DATASET_ID, COL_SUBJECT_ID, COL_SESSION_ID, COL_TRIAL_ID]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Daten laden und mit Metadaten mergen
df = pd.read_csv(os.path.join(DATA_PATH, RAW_FORCE_FILE))
print(f"Rohdaten geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

metadata = pd.read_csv(os.path.join(DATA_PATH, METADATA_FILE))
merge_cols = [c for c in [COL_DATASET_ID, COL_SUBJECT_ID, COL_SESSION_ID, COL_TRIAL_ID]
              if c in df.columns and c in metadata.columns]
df = df.merge(metadata, on=merge_cols, how='left')
print(f"Nach Metadaten-Merge: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

Rohdaten geladen: 8819 Zeilen, 105 Spalten
Nach Metadaten-Merge: 8819 Zeilen, 123 Spalten


In [ ]:
# Merge der Kraftrichtungen (AP, ML)
for force_file, prefix in [(AP_FORCE_FILE, 'F_AP_PRO_'), (ML_FORCE_FILE, 'F_ML_PRO_')]:
    file_path = os.path.join(DATA_PATH, force_file)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} nicht gefunden, aber {prefix}* wird benötigt.")

    extra = pd.read_csv(file_path)
    force_cols = [c for c in extra.columns if c.startswith(prefix)]
    if not force_cols:
        raise ValueError(f"{file_path} enthält keine Spalten mit Präfix '{prefix}'.")

    merge_cols = [c for c in [COL_DATASET_ID, COL_SUBJECT_ID, COL_SESSION_ID, COL_TRIAL_ID]
                  if c in df.columns and c in extra.columns]
    df = df.merge(extra[merge_cols + force_cols], on=merge_cols, how='left')
    print(f"{force_file} gemerged: +{len(force_cols)} Spalten ({prefix}*)")

n_v = sum(c.startswith('F_V_PRO_') for c in df.columns)
n_ap = sum(c.startswith('F_AP_PRO_') for c in df.columns)
n_ml = sum(c.startswith('F_ML_PRO_') for c in df.columns)
print(f"Kraftrichtungs-Spalten: F_V_PRO={n_v}, F_AP_PRO={n_ap}, F_ML_PRO={n_ml}")

Datenbereinigung - Inhalte ausschließen

In [ ]:
# Subjects without sex excluding
n_before = df[COL_SUBJECT_ID].nunique()
missing_sex_subjects = set(df.loc[df[COL_SEX].isna(), COL_SUBJECT_ID].unique())
df = df[~df[COL_SUBJECT_ID].isin(missing_sex_subjects)].reset_index(drop=True)
print(f"Subjekte ohne SEX: {len(missing_sex_subjects)} ausgeschlossen "
      f"({n_before} -> {df[COL_SUBJECT_ID].nunique()})")


#subjects with conflicting sex data excluding
demo_cols = [COL_AGE, COL_SEX, COL_HEIGHT, COL_BODY_MASS]
n_unique_per_subject = df.groupby(COL_SUBJECT_ID)[demo_cols].nunique()

for col in demo_cols:
    n_affected = int((n_unique_per_subject[col] > 1).sum())
    print(f"{col}: {n_affected} Subjekt(e) mit >1 unterschiedlichem Wert über Sessions")

sex_conflict_subjects = set(n_unique_per_subject[n_unique_per_subject[COL_SEX] > 1].index)

n_before = df[COL_SUBJECT_ID].nunique()
df = df[~df[COL_SUBJECT_ID].isin(sex_conflict_subjects)].reset_index(drop=True)
print(f"Subjekte mit SEX-Konflikt: {len(sex_conflict_subjects)} ausgeschlossen "
      f"({n_before} -> {df[COL_SUBJECT_ID].nunique()})")

# Probanden mit NaN Age oder Height ausschließen
n_before = df[COL_SUBJECT_ID].nunique()
missing_demo_subjects = set()
for col in [COL_AGE, COL_HEIGHT]:
    missing_per_subject = df.groupby(COL_SUBJECT_ID)[col].apply(lambda x: x.isna().all())
    subjects_missing_this_col = set(missing_per_subject[missing_per_subject].index)
    print(f"{col}: {len(subjects_missing_this_col)} Subjekt(e) komplett ohne Wert")
    missing_demo_subjects |= subjects_missing_this_col

df = df[~df[COL_SUBJECT_ID].isin(missing_demo_subjects)].reset_index(drop=True)
print(f"Subjekte ohne AGE/HEIGHT: {len(missing_demo_subjects)} ausgeschlossen "
      f"({n_before} -> {df[COL_SUBJECT_ID].nunique()})")

In [ ]:
# DIES IST VERALTET UND WIRD NICHT MEHR VERWENDET, DA DIE SEX KONFLIKTE BEREITS "SICHER" GELÖST WORDEN SIND INDEM MAN SIE AUSSCHLIESST, IN ABSPRACHE MIT MANFRED RÖSSLE
# IST NUR ALS DOKUIMENTATIONSZWECK DRIN

# Da Gutenberg Gait Datase eine fehlerhafte Geschlechtsenkodierung hat, wird hier der SEX wert des ersten SSESSION_TYPE des Probanden genommen
# WICHTIG: DIES IST IN ABSPRACHE MIT DEM GRÜNDER/AUTHOR DES DATENSATZES GESCHEHEN UND IST DER WAHRHEITSWERT GEMäß FABIAN HORST
""" 
demo_cols = [COL_AGE, COL_SEX, COL_HEIGHT, COL_BODY_MASS]
df_sorted = df.sort_values([COL_SUBJECT_ID, COL_SESSION_ID]).copy()

df_sorted['_is_baseline'] = df_sorted[COL_SESSION_TYPE] == BASELINE_SESSION_TYPE_VALUE
df_sorted = df_sorted.sort_values(
    [COL_SUBJECT_ID, '_is_baseline', COL_SESSION_ID], ascending=[True, False, True]
)

first_non_null = lambda x: x.dropna().iloc[0]
subject_df = df_sorted.groupby(COL_SUBJECT_ID, as_index=False)[demo_cols].agg(first_non_null)

n_subjects = len(subject_df)
print(f"{n_subjects} eindeutige Subjekte in der Subjekt-Tabelle.")
"""

In [ ]:
# Sex Label Encoding ist float -> wird auf int umgeändert. (0 = female, 1 = male) siehe Table 4 Gutenberg Gait Dataset

print(f"SEX-Werte bevor Enkodierung: {df['SEX'].unique()}")
print(f"SEX-Datentypbevor Enkodierung:: {df['SEX'].dtype}")

subject_df['SEX_LABEL'] = subject_df[COL_SEX].astype(int)
sex_classes = sorted(int(v) for v in subject_df[COL_SEX].unique())

print(f"SEX-Werte nach Enkodierung: {df['SEX'].unique()}")
print(f"SEX-Datentyp nach Enkodierung:: {df['SEX'].dtype}")


## Balancierte AGE und Height Brackets bilden
Berechnen von automatischen Quartilen

In [ ]:
# Age Brackets erstellen
age_values = subject_df[COL_AGE]

age_bin_edges, n_age_brackets = None, None
for k in range(AGE_MAX_BRACKETS, 1, -1):
    _, edges = pd.qcut(age_values, q=k, retbins=True, duplicates='drop')
    if len(edges) - 1 != k:
        continue
    counts = pd.cut(age_values, bins=edges, include_lowest=True).value_counts()
    if counts.min() >= AGE_MIN_SUBJECTS_PER_BRACKET:
        age_bin_edges, n_age_brackets = edges, k
        break

if age_bin_edges is None:
    median = age_values.median()
    age_bin_edges = np.array([age_values.min() - 1e-9, median, age_values.max() + 1e-9])
    n_age_brackets = 2

age_labels = [f"{round(age_bin_edges[i], 1)}-{round(age_bin_edges[i + 1], 1)}y"
              for i in range(len(age_bin_edges) - 1)]
print(f"{n_age_brackets} Age-Brackets: {age_labels}")

subject_df['AGE_BRACKET'] = pd.cut(age_values, bins=age_bin_edges, include_lowest=True, labels=age_labels)
subject_df['AGE_BRACKET_LABEL'] = LabelEncoder().fit_transform(subject_df['AGE_BRACKET'])

print(subject_df['AGE_BRACKET'].value_counts().to_string())

In [ ]:
# Height Brackets erstellen
height_values = subject_df[COL_HEIGHT]

height_bin_edges, n_height_brackets = None, None
for k in range(HEIGHT_MAX_BRACKETS, 1, -1):
    _, edges = pd.qcut(height_values, q=k, retbins=True, duplicates='drop')
    if len(edges) - 1 != k:
        continue
    counts = pd.cut(height_values, bins=edges, include_lowest=True).value_counts()
    if counts.min() >= HEIGHT_MIN_SUBJECTS_PER_BRACKET:
        height_bin_edges, n_height_brackets = edges, k
        break

if height_bin_edges is None:
    median = height_values.median()
    height_bin_edges = np.array([height_values.min() - 1e-9, median, height_values.max() + 1e-9])
    n_height_brackets = 2

height_labels = [f"{round(height_bin_edges[i], 1)}-{round(height_bin_edges[i + 1], 1)}cm"
                 for i in range(len(height_bin_edges) - 1)]
print(f"{n_height_brackets} Height-Brackets: {height_labels}")

subject_df['HEIGHT_BRACKET'] = pd.cut(height_values, bins=height_bin_edges, include_lowest=True, labels=height_labels)
subject_df['HEIGHT_BRACKET_LABEL'] = LabelEncoder().fit_transform(subject_df['HEIGHT_BRACKET'])

print(subject_df['HEIGHT_BRACKET'].value_counts().to_string())

Train/Test-Split vorbereiten

In [ ]:
# Train/ Test Split erstellen - Auf subject Ebene
train_df, test_df = train_test_split(
    subject_df, test_size=TEST_SIZE, 
    random_state=RANDOM_STATE, 
    stratify=subject_df['SEX_LABEL'] # Ermöglicht uns Bias zu vermeiden, indem wir gleiche Verteilung von Geschlecht in Train und Test haben. Minimiert Overfitting auf Geschlecht.
)

train_subjects = set(train_df[COL_SUBJECT_ID])
test_subjects = set(test_df[COL_SUBJECT_ID])

print(f"Train: {len(train_subjects)} Subjekte, Test: {len(test_subjects)} Subjekte")

subject_df['SPLIT'] = np.where(subject_df[COL_SUBJECT_ID].isin(train_subjects), 'train', 'test')

split_distribution = {}
for col in ['SEX', 'AGE_BRACKET', 'HEIGHT_BRACKET']:
    ct = pd.crosstab(subject_df[col], subject_df['SPLIT'])
    split_distribution[col] = {split_name: {str(k): int(v) for k, v in counts.items()}
                                for split_name, counts in ct.to_dict().items()}
    print(f"\n-- {col} --")
    print(ct.to_string())

In [ ]:
# Split + Labels zurück auf Trial-Ebene mergen und speichern
merge_cols = [COL_SUBJECT_ID, 'SEX_LABEL', 'AGE_BRACKET', 'AGE_BRACKET_LABEL',
              'HEIGHT_BRACKET', 'HEIGHT_BRACKET_LABEL', 'SPLIT']
df = df.merge(subject_df[merge_cols], on=COL_SUBJECT_ID, how='left')

prepared_path = os.path.join(OUTPUT_DIR, PREPARED_DATA_FILE)
df.to_csv(prepared_path, index=False) # als CSV speichern
print(f"Prepared data gespeichert: {prepared_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")

Split-Metadaten speichern (`split_meta.json`)

In [ ]:
split_meta = {
    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'n_subjects_total': n_subjects,
    'n_subjects_train': len(train_subjects),
    'n_subjects_test': len(test_subjects),
    'train_subject_ids': sorted(int(s) for s in train_subjects),
    'test_subject_ids': sorted(int(s) for s in test_subjects),
    'sex_classes': sex_classes,
    'age_bracket_edges': [float(x) for x in age_bin_edges],
    'age_bracket_labels': age_labels,
    'height_bracket_edges': [float(x) for x in height_bin_edges],
    'height_bracket_labels': height_labels,
    'split_distribution': split_distribution,
    'excluded_subjects': {
        'missing_sex': sorted(int(s) for s in missing_sex_subjects),
        'sex_conflict_between_sessions': sorted(int(s) for s in sex_conflict_subjects),
        'missing_age_or_height': sorted(int(s) for s in missing_demo_subjects),
    },
}

meta_path = os.path.join(OUTPUT_DIR, SPLIT_META_FILE)
with open(meta_path, 'w') as f:
    json.dump(split_meta, f, indent=2)
print(f"Split-Metadaten gespeichert: {meta_path}")

print("\nDONE!")